# 12 · Powerlifting：最终冲刺——更多全量重训 + TabM 近全量（Colab GPU）

**11 的结果**

|模型（5 折 × 3 种子）|三种子平均 OOF|NNLS 权重|
|---|---:|---:|
|cb_rel（DOTS 相对力量 CatBoost）|**76.784**|0.453|
|TabM|77.050|0.217|
|CatBoost|77.097|0.042|
|lgb_rel|77.167|0.057|
|LightGBM|77.200|0.031|
|XGBoost|77.693|0.201|

融合（嵌套）76.395 → 双胞胎 76.172 → S3 平滑 76.044（交叉拟合）。公开榜：**全量重训 76.360**、混合 76.421、折平均 76.520。全量重训比折平均好 0.16，说明**训练数据越多越好**。11 的数据清洗实验：训练时去掉疑似行，LightGBM +0.043、XGBoost +0.063、CatBoost +0.071（都变差），保持不删。

**本 notebook 做三件事**（不重跑交叉验证，融合权重和第二层系数沿用 11）：

|步骤|内容|为什么|
|---|---|---|
|1. 学习率检查|cb_rel 用学习率 0.03（原 0.05），种子 42 五折，与 11 同折配对比较；平均 ≤ −0.02 且 ≥ 4/5 折更好才在后面的全量重训里使用|cb_rel 占融合 45%；更小学习率通常更稳|
|2. 更多全量重训种子|cb_rel、XGBoost、lgb_rel、LightGBM 各再加 5 个种子（共 8 个）|全量重训的测试预测目前只平均了 3 个种子|
|3. TabM 近全量|TabM 在 95% 训练行上训练、5% 早停，3 个种子|11 的「全量版」里 TabM 只有折模型（每个只用 72% 的数据），而 TabM 占融合 22%|

**输出**（`artifacts/final_v2/`）：
- `submission_final_v2_seeds_final.csv`：只加更多种子（TabM 仍用折平均）
- `submission_final_v2_seeds_tabmfull_final.csv`：再加 TabM 近全量（**预计最好**）

两个都交，和 11 的 76.360 配对比较。

**操作**：Colab 选 **T4 GPU**，从上到下运行；约 50–60 分钟，可续跑。需要 11 的结果目录 `artifacts/final_v1/` 完整。

## 1. 环境与配置（与 11 相同，实验名不同）

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import inspect
import json
import os
import shutil
import subprocess
import sys
import time
import warnings

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'PL_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting',
)).expanduser().resolve()
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

CATBOOST_VERSION = '1.2.10'
to_install = []
try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启运行时后从头运行')
    to_install.append(f'catboost=={CATBOOST_VERSION}')
try:
    importlib.metadata.version('pytabkit')
except importlib.metadata.PackageNotFoundError:
    to_install.append('pytabkit')
if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
    importlib.invalidate_caches()

import catboost
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from IPython.display import display
from scipy.optimize import nnls
from sklearn.model_selection import KFold, train_test_split

try:
    from pytabkit import TabM_D_Regressor
    PYTABKIT_ERROR = None
except Exception as exc:                       # 安装或导入失败时其余模型照常运行
    TabM_D_Regressor = None
    PYTABKIT_ERROR = repr(exc)

warnings.filterwarnings('ignore', category=UserWarning)
if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')
if shutil.which('nvidia-smi') is None or not torch.cuda.is_available():
    raise RuntimeError('没有可用的 GPU：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          text=True, capture_output=True, check=False).stdout.strip().splitlines()[0]

EXPERIMENT_ID = 'final_v2'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / EXPERIMENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42                                  # 07–10 的五折划分种子（第一个种子）
SEEDS = [42, 7, 2024]                      # 每个种子：不同的五折划分、内层早停划分和模型随机性
N_FOLDS = 5
INNER_STOP_FRACTION = 0.10
MAX_ROUNDS = 20000
FINAL_MODELS = ['lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel', 'tabm']   # 每折内的运行顺序：快的先跑
FULL_REFIT = True                          # 树模型在全部训练行上重训
REFIT_FACTOR = 1.2                         # 全量重训轮数 = CV 平均最佳轮数 × 1.2（数据多约 39%）
RUN_CLEANING_TEST = True                   # 第 9 节：数据清洗实验（只作报告证据）
MODELS = ['lgb', 'xgb', 'catboost']        # 09 / 10 特征单元使用的三个模型族名
RUN_ONLY = None                            # 09 / 10 特征单元需要的变量，这里不用
KEEP_DELTA = -0.05
EXPECTED_SHA256 = {'train.csv': '62319814116506fb95a51f1185dbacdb170ab055642ee4c347a30584093d0856', 'test.csv': '563c6fa29f9e535740ef1f09b25954f00fbe242c81c0e5a7343dc18aa962ba46'}
VERSIONS = {'catboost': catboost.__version__, 'xgboost': xgb.__version__, 'lightgbm': lgb.__version__,
            'torch': torch.__version__, 'pytabkit': None if PYTABKIT_ERROR else importlib.metadata.version('pytabkit'),
            'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}
print('PROJECT_DIR :', PROJECT_DIR)
print('OUTPUT_DIR  :', OUTPUT_DIR)
print('GPU         :', GPU_NAME)
print('版本         :', VERSIONS)
if PYTABKIT_ERROR:
    print('pytabkit 不可用，TabM 会在预检中跳过：', PYTABKIT_ERROR)

PREV_DIR = PROJECT_DIR / 'artifacts' / 'final_v1'          # 11 的结果
PREV_SEEDS = [42, 7, 2024]
EXTRA_SEEDS = [11, 23, 57, 99, 123]                         # 全量重训新增的种子
EXTRA_MODELS = ['lgb', 'xgb', 'lgb_rel', 'cb_rel']          # 快的先跑
TABM_FULL_SEEDS = [42, 7, 2024]
TABM_HOLDOUT = 0.05                                         # TabM 近全量：留 5% 早停
RUN_LR_CHECK = True
if not (PREV_DIR / 'summary_weights.csv').is_file() or not (PREV_DIR / 'oof_nested_blend.npy').is_file():
    raise FileNotFoundError(f'缺少 11 的结果：{PREV_DIR}；请先完整运行 11')

Mounted at /content/drive
PROJECT_DIR : /content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting
OUTPUT_DIR  : /content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting/artifacts/final_v2
GPU         : Tesla T4, 15360 MiB
版本         : {'catboost': '1.2.10', 'xgboost': '3.4.1', 'lightgbm': '4.6.0', 'torch': '2.11.0+cu130', 'pytabkit': '1.7.3', 'numpy': '2.1.3', 'pandas': '2.2.3', 'python': '3.13.15'}


## 2. 读取并核对数据（与 07 相同）

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rmse(a, b):
    a, b = np.asarray(a, dtype='float64'), np.asarray(b, dtype='float64')
    return float(np.sqrt(np.mean((a - b) ** 2)))


for name, expected in EXPECTED_SHA256.items():
    require(sha256_file(PROJECT_DIR / name) == expected, f'{name} 与本地版本不同；请检查 Drive 上的数据')

TARGET = 'TotalKg'
train = pd.read_csv(PROJECT_DIR / 'train.csv')
test = pd.read_csv(PROJECT_DIR / 'test.csv')
RAW_NUM = ['Age', 'BodyweightKg', 'MeetYear']
RAW_FEATURES = [c for c in test.columns if c != 'id']
CAT = [c for c in RAW_FEATURES if c not in RAW_NUM]          # 不依赖 object / str dtype（pandas 2 与 3 都可用）
require(train.shape == (105000, 18) and test.shape == (45000, 17), '数据形状不符')
require(len(CAT) == 13 and all(pd.api.types.is_numeric_dtype(train[c]) for c in RAW_NUM)
        and not any(pd.api.types.is_numeric_dtype(train[c]) for c in CAT), '列分组不符')
require(not train[RAW_FEATURES].isna().any().any() and not test[RAW_FEATURES].isna().any().any(), '出现空值')
y = train[TARGET].to_numpy(dtype='float64')
n_train = len(train)

FOLD_IDS = np.zeros(n_train, dtype=int)
for k, (_, valid) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.arange(n_train)), start=1):
    FOLD_IDS[valid] = k
print(f'train {train.shape}  test {test.shape}；每折验证行数：{np.bincount(FOLD_IDS)[1:].tolist()}')

train (105000, 18)  test (45000, 17)；每折验证行数：[21000, 21000, 21000, 21000, 21000]


## 3. 第一轮特征（与 07 相同）

In [3]:
both = pd.concat([train[RAW_FEATURES], test[RAW_FEATURES]], ignore_index=True)

# F0：v2 的体重级别特征
weight_class = pd.to_numeric(both['WeightClassKg'].str.replace('+', '', regex=False), errors='coerce')
F0_EXTRA = pd.DataFrame({
    'WeightClass_num': weight_class,
    'Is_SHW': both['WeightClassKg'].str.endswith('+').astype('float64'),
    'BW_minus_Class': both['BodyweightKg'] - weight_class,
})

# P：记录精度（树按阈值切数值，学不到「是不是 .5」「是不是整数」）
age2 = (both['Age'] * 2).round()
require(np.allclose(both['Age'] * 2, age2), 'Age 不是 0.5 的整数倍')
bw = both['BodyweightKg']
is_int = np.isclose(bw, bw.round())
is_1dec = np.isclose(bw * 10, (bw * 10).round())
is_half = is_1dec & ~is_int & np.isclose((bw * 10).round() % 10, 5)
pounds = bw / 0.45359237
P = pd.DataFrame({
    'age_half': (age2 % 2).astype('float64'),
    'bw_precision': np.select([is_int, is_half, is_1dec], [0.0, 1.0, 2.0], default=3.0),
    'bw_lb_int': (np.abs(pounds - pounds.round()) < 0.015).astype('float64'),
})

# T：跨地参赛（居住地未知时为 NaN）
abroad = (both['Country'] != both['MeetCountry']).astype('float64').where(both['Country'] != 'Unknown')
state_known = ((both['State'] != 'Unknown') & (both['MeetState'] != 'Unknown')
               & (both['Country'] == both['MeetCountry']))
other_state = (both['State'] != both['MeetState']).astype('float64').where(state_known)
T = pd.DataFrame({'abroad': abroad, 'other_state': other_state})

# M：比赛背景；「比赛」≈ 联盟 × 年份 × 比赛国 × 比赛州（train+test 无标签统计）
meet = (both['Federation'] + '|' + both['MeetYear'].astype(str) + '|'
        + both['MeetCountry'] + '|' + both['MeetState'])


def by_meet(values):
    return values.groupby(meet).transform('mean')


M = pd.DataFrame({
    'meet_n': meet.map(meet.value_counts()).astype('float64'),
    'meet_sexeq_n': both.groupby([meet, both['Sex'], both['Equipment']])['Age'].transform('size').astype('float64'),
    'fed_year_n': both.groupby(['Federation', 'MeetYear'])['Age'].transform('size').astype('float64'),
    'meet_share_other_state': by_meet(other_state),
    'meet_share_abroad': by_meet(abroad),
    'meet_share_age_half': by_meet(P['age_half']),
    'meet_share_equipped': by_meet((both['Equipment'] != 'Raw').astype('float64')),
    'meet_share_male': by_meet((both['Sex'] == 'M').astype('float64')),
    'meet_mean_age': by_meet(both['Age']),
})

BLOCKS = {'P': P, 'T': T, 'M': M}
VARIANTS = {'F0': [], 'P': ['P'], 'T': ['T'], 'M': ['M'], 'PT': ['P', 'T'], 'ALL': ['P', 'T', 'M']}
for name, frame in {'F0': F0_EXTRA, **BLOCKS}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')

_FRAMES = {}


def frames_for(variant, model):
    """(train, test) 特征表。CatBoost 的类别为字符串；LGB / XGB 为 category（词表来自 train+test，无标签）。"""
    kind = 'str' if model == 'catboost' else 'category'
    if (variant, kind) not in _FRAMES:
        frame = pd.concat([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in VARIANTS[variant]] + [both[CAT]], axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        frame[CAT] = frame[CAT].astype(str).astype(object) if kind == 'str' else frame[CAT].astype('category')
        _FRAMES[(variant, kind)] = (frame.iloc[:n_train].reset_index(drop=True),
                                    frame.iloc[n_train:].reset_index(drop=True))
    return _FRAMES[(variant, kind)]


print({v: frames_for(v, 'lgb')[0].shape[1] for v in VARIANTS})
summary_new = pd.concat([P, T, M], axis=1)
display(pd.DataFrame({'train_mean': summary_new.iloc[:n_train].mean(), 'test_mean': summary_new.iloc[n_train:].mean(),
                      'nan_share': summary_new.isna().mean()}).round(4))

{'F0': 19, 'P': 22, 'T': 21, 'M': 28, 'PT': 24, 'ALL': 33}


,train_mean,test_mean,nan_share
age_half,0.4817,0.4782,0.0000
bw_precision,1.9668,1.9609,0.0000
bw_lb_int,0.0500,0.0503,0.0000
abroad,0.0824,0.0822,0.0184
other_state,0.2670,0.2724,0.6491
meet_n,63.6623,63.9177,0.0000
meet_sexeq_n,33.0578,33.1368,0.0000
fed_year_n,653.6460,656.8197,0.0000
meet_share_other_state,0.2661,0.2673,0.6219
meet_share_abroad,0.0833,0.0837,0.0004


## 4. 第二轮特征（与 09 相同）

In [4]:
_FRAMES.clear()                     # 07 的变体表不再需要，释放内存
CORE9 = ['Age', 'BodyweightKg', 'MeetYear', 'Sex', 'Federation', 'MeetCountry', 'MeetState', 'Country', 'State']
sex, equipment = both['Sex'], both['Equipment']
state = both['State'].where(both['State'] != 'Unknown')
country = both['Country'].where(both['Country'] != 'Unknown')
national = both['Federation'] + '|' + both['MeetYear'].astype(str) + '|' + both['MeetCountry']


def counts(key):
    return key.map(key.value_counts()).astype('float64')


def n_distinct(values, key):
    return values.groupby(key).transform('nunique').astype('float64')


# M2：比赛背景加强（meet 与 by_meet 来自 07 的特征单元）
M2 = pd.DataFrame({
    'meet_n_states': n_distinct(state, meet),
    'meet_n_countries': n_distinct(country, meet),
    'meet_n_wclass': n_distinct(both['WeightClassKg'], meet),
    'meet_share_tested': by_meet((both['Tested'] == 'Yes').astype('float64')),
    'meet_share_raw': by_meet((equipment == 'Raw').astype('float64')),
    'meet_share_multi': by_meet((equipment == 'Multi-ply').astype('float64')),
    'meet_share_bw2dec': by_meet((P['bw_precision'] == 3).astype('float64')),
    'meet_share_junior': by_meet((both['Age'] < 24).astype('float64')),
    'meet_share_master': by_meet((both['Age'] >= 40).astype('float64')),
    'meet_age_std': both['Age'].groupby(meet).transform('std'),
    'meet_sex_bw_mean': both['BodyweightKg'].groupby([meet, sex]).transform('mean'),
    'nat_n': counts(national),
    'nat_n_meetstates': n_distinct(both['MeetState'], national),
    'nat_n_states': n_distinct(state, national),
})

# D：组别背景
division = meet + '|' + sex + '|' + equipment + '|' + both['WeightClassKg']
age_division = meet + '|' + sex + '|' + equipment + '|' + both['AgeClass']
exact_key = pd.Series(pd.util.hash_pandas_object(both[RAW_FEATURES], index=False).to_numpy())
core_key = pd.Series(pd.util.hash_pandas_object(both[CORE9], index=False).to_numpy())
D = pd.DataFrame({
    'div_n': counts(division),
    'div_bw_rank': both['BodyweightKg'].groupby(division).rank(pct=True),
    'div_age_rank': both['Age'].groupby(division).rank(pct=True),
    'agediv_n': counts(age_division),
    'dup_exact_n': counts(exact_key) - 1,
    'dup_core_n': counts(core_key) - 1,
})

# H：近似同一运动员的历史（出生年 = floor(MeetYear − Age)：x.5 年龄时恰为出生年，整数年龄可能差 1）
valid = (both['Country'] != 'Unknown').to_numpy()
birth = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int).astype(str)
athlete = sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation'] + '|' + birth
athlete0 = sex + '|' + both['Country'] + '|' + both['State'] + '|' + birth
year = both['MeetYear'].astype('float64')
H = pd.DataFrame({
    'ath_n': counts(athlete),
    'ath_n_years': n_distinct(year, athlete),
    'ath_years_since_first': year - year.groupby(athlete).transform('min'),
    'ath_years_to_last': year.groupby(athlete).transform('max') - year,
    'ath_n_equipment': n_distinct(equipment, athlete),
    'ath_bw_delta': both['BodyweightKg'] - both['BodyweightKg'].groupby(athlete).transform('mean'),
    'ath0_n': counts(athlete0),
    'ath0_n_feds': n_distinct(both['Federation'], athlete0),
})
H.loc[~valid] = np.nan


# S：DOTS 系数（TotalKg = DOTS 分 × dots_scale）
def dots_scale(sex_values, bw_values):
    s = np.asarray(sex_values).astype(str)
    bw = np.asarray(bw_values, dtype='float64')
    m, f = np.clip(bw, 40, 210), np.clip(bw, 40, 150)
    poly_m = -1.0930e-6 * m**4 + 7.391293e-4 * m**3 - 0.1918759221 * m**2 + 24.0900756 * m - 307.75076
    poly_f = -1.0706e-6 * f**4 + 5.158568e-4 * f**3 - 0.1126655495 * f**2 + 13.6175032 * f - 57.96288
    return np.where(s == 'M', poly_m, np.where(s == 'F', poly_f, 0.5 * (poly_m + poly_f))) / 500.0


S = pd.DataFrame({'dots_scale': dots_scale(sex, both['BodyweightKg'])})

# C：只给 CatBoost 的高基数 ID（有序目标统计编码）
C = pd.DataFrame({'meet_id': meet, 'athlete_id': (athlete + '|' + equipment).where(valid, 'unknown')})

BLOCKS2 = {'M2': M2, 'D': D, 'H': H, 'S': S}
CB_ONLY = {'C': C}
for name, frame in {**BLOCKS2, **CB_ONLY}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
for name, frame in BLOCKS2.items():
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
require(S['dots_scale'].min() > 0.3, 'DOTS 系数异常')

R1_BLOCKS = ['P', 'T', 'M']
VARIANT_BLOCKS = {'R1': [], 'M2': ['M2'], 'D': ['D'], 'H': ['H'], 'S': ['S'], 'C': ['C']}
MODELS_FOR = {'C': ['catboost']}      # C 只改变 CatBoost；融合时 LightGBM / XGBoost 用 R1
SCREEN = [v for v in VARIANT_BLOCKS if RUN_ONLY is None or v in RUN_ONLY or v == 'R1']
_FRAMES2 = {}


def models_for(variant):
    return MODELS_FOR.get(variant, MODELS)


def frames2(variant, model):
    """(train, test, 类别列)。R1 = F0 + P + T + M；C 块只加给 CatBoost。"""
    blocks = VARIANT_BLOCKS[variant]
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES2:
        nums = ([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in R1_BLOCKS]
                + [BLOCKS2[b] for b in blocks if b in BLOCKS2])
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES2[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES2[key]


new_cols = pd.concat([M2, D, H, S], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print('C 块类别数：', {c: int(C[c].nunique()) for c in C.columns})
print('各变体列数（LightGBM / XGBoost）：', {v: frames2(v, 'lgb')[0].shape[1] for v in VARIANT_BLOCKS if v != 'C'},
      '｜CatBoost C：', frames2('C', 'catboost')[0].shape[1])

,train_mean,test_mean,nan_share
meet_n_states,3.2039,3.2371,0.0000
meet_n_countries,2.0094,2.0170,0.0000
meet_n_wclass,10.7194,10.7360,0.0000
meet_share_tested,0.7627,0.7604,0.0000
meet_share_raw,0.6208,0.6212,0.0000
meet_share_multi,0.0234,0.0240,0.0000
meet_share_bw2dec,0.2949,0.2969,0.0000
meet_share_junior,0.4549,0.4538,0.0000
meet_share_master,0.1703,0.1714,0.0000
meet_age_std,10.0628,10.0618,0.0104


C 块类别数： {'meet_id': 10546, 'athlete_id': 46893}
各变体列数（LightGBM / XGBoost）： {'R1': 33, 'M2': 47, 'D': 39, 'H': 41, 'S': 34} ｜CatBoost C： 35


## 5. 第三轮特征与各模型的最终特征表（与 11 相同）

In [5]:
_FRAMES2.clear()                    # 09 的变体表不再需要，释放内存
birth_num = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int)


def fuzzy_count(prefix_key):
    """同前缀、出生年为 b−1、b、b+1 的行数（整数年龄推算的出生年可能差 1）。"""
    exact = prefix_key + '|' + birth_num.astype(str)
    table = exact.value_counts()
    total = exact.map(table).astype('float64')
    for d in (-1, 1):
        total = total + (prefix_key + '|' + (birth_num + d).astype(str)).map(table).fillna(0)
    return total


# H2：运动员生涯（athlete、valid 来自 09 的特征单元；M、T 来自 07；M2 来自 09）
bw = both['BodyweightKg']
H2 = pd.DataFrame({
    'ath_max_meet_states': M2['meet_n_states'].groupby(athlete).transform('max'),
    'ath_mean_meet_states': M2['meet_n_states'].groupby(athlete).transform('mean'),
    'ath_max_meet_n': M['meet_n'].groupby(athlete).transform('max'),
    'ath_max_nat_states': M2['nat_n_states'].groupby(athlete).transform('max'),
    'ath_ever_abroad': T['abroad'].groupby(athlete).transform('max'),
    'ath_n_wclass': n_distinct(both['WeightClassKg'], athlete),
    'ath_bw_range': bw.groupby(athlete).transform('max') - bw.groupby(athlete).transform('min'),
    'ath_first_age': both['Age'].groupby(athlete).transform('min'),
    'ath_n_same_year': both.groupby([athlete, both['MeetYear']])['Age'].transform('size').astype('float64'),
    'ath_share_tested': (both['Tested'] == 'Yes').astype('float64').groupby(athlete).transform('mean'),
    'ath_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation']),
    'ath0_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State']),
})
H2.loc[~valid] = np.nan

# E：比赛的经验结构
experience = H['ath_years_since_first']
E = pd.DataFrame({
    'meet_mean_ath_n': H['ath_n'].groupby(meet).transform('mean'),
    'meet_mean_ath_exp': experience.groupby(meet).transform('mean'),
    'meet_share_veterans': (experience >= 3).astype('float64').where(experience.notna()).groupby(meet).transform('mean'),
    'meet_share_newcomers': (H['ath_n'] == 1).astype('float64').where(H['ath_n'].notna()).groupby(meet).transform('mean'),
})

ROUND3 = {'H2': H2, 'E': E}
for name, frame in ROUND3.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
NUMERIC_BLOCKS = {**BLOCKS, **BLOCKS2, **ROUND3}          # P T M（07）、M2 D H S（09）、H2 E（本轮）
R2_BLOCKS = ['P', 'T', 'M', 'H']                          # 09 的 ALL2
EXTRAS = {'lgb': [], 'xgb': ['M2', 'S', 'D'], 'catboost': ['M2', 'C']}   # PM：09 中各模型单独有效的块
VARIANTS3 = {'R2': [], 'H2': ['H2'], 'E': ['E'], 'PM': ['PM']}
SCREEN3 = [v for v in VARIANTS3 if RUN_ONLY is None or v in RUN_ONLY or v == 'R2']
_FRAMES3 = {}


def blocks_for(variant, model):
    out = list(R2_BLOCKS)
    for b in VARIANTS3[variant]:
        out += EXTRAS[model] if b == 'PM' else [b]
    return out


def runs(model, variant):
    """该模型在这个变体上的特征与 R2 不同才需要训练（例如 PM 对 LightGBM 没有变化）。"""
    return variant == 'R2' or blocks_for(variant, model) != blocks_for('R2', model)


def frames3(variant, model):
    """(train, test, 类别列)。"""
    blocks = blocks_for(variant, model)
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES3:
        nums = [both[RAW_NUM], F0_EXTRA] + [NUMERIC_BLOCKS[b] for b in blocks if b in NUMERIC_BLOCKS]
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} {model} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES3[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES3[key]


new_cols = pd.concat([H2, E], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print({f'{v}/{m}': frames3(v, m)[0].shape[1] for v in VARIANTS3 for m in MODELS if runs(m, v)})

# ---- 最终特征：10 的 ALL3（三个块都达标）----
_FRAMES3.clear()
VARIANTS3['ALL3'] = ['H2', 'E', 'PM']
FAMILY = {'lgb': 'lgb', 'lgb_rel': 'lgb', 'tabm': 'lgb', 'xgb': 'xgb', 'catboost': 'catboost', 'cb_rel': 'catboost'}
FRAMES = {m: frames3('ALL3', FAMILY[m]) for m in FINAL_MODELS}
for m, (Xtr_m, Xte_m, cats_m) in FRAMES.items():
    print(f'{m:9s} {Xtr_m.shape[1]} 列（{len(cats_m)} 个类别列），块：{blocks_for("ALL3", FAMILY[m])}')

,train_mean,test_mean,nan_share
ath_max_meet_states,7.2553,7.2965,0.0184
ath_mean_meet_states,3.2682,3.2702,0.0184
ath_max_meet_n,112.4144,112.6788,0.0184
ath_max_nat_states,17.7174,17.7553,0.0184
ath_ever_abroad,0.1208,0.1206,0.0184
ath_n_wclass,6.3779,6.4201,0.0184
ath_bw_range,48.6372,49.1032,0.0184
ath_first_age,24.1705,24.0925,0.0184
ath_n_same_year,4.9418,4.9522,0.0184
ath_share_tested,0.7628,0.7612,0.0184


{'R2/lgb': 41, 'R2/xgb': 41, 'R2/catboost': 41, 'H2/lgb': 53, 'H2/xgb': 53, 'H2/catboost': 53, 'E/lgb': 45, 'E/xgb': 45, 'E/catboost': 45, 'PM/xgb': 62, 'PM/catboost': 57}
lgb       57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']
xgb       78 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'S', 'D']
lgb_rel   57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']
catboost  73 列（15 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'C']
cb_rel    73 列（15 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'C']
tabm      57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']


## 6. 模型定义（与 11 相同）与预检

In [6]:
LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, cat_smooth=20, verbose=-1)
XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                  reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                  max_cat_to_onehot=1, eval_metric='rmse')
CB_PARAMS = dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                 border_count=128)
TABM_PARAMS = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0, num_emb_type='pwl')
TABM_FALLBACK = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0)
PATIENCE = {'lgb': 200, 'lgb_rel': 200, 'xgb': 200, 'catboost': 300, 'cb_rel': 300}
GBDT = [m for m in FINAL_MODELS if m != 'tabm']

SCALE = dots_scale(train['Sex'], train['BodyweightKg'])           # dots_scale 来自 09 的特征单元
SCALE_TEST = dots_scale(test['Sex'], test['BodyweightKg'])
require(min(SCALE.min(), SCALE_TEST.min()) > 0.3, 'DOTS 系数异常')

X_NN, XT_NN, CATS_NN = FRAMES['tabm']
NUM_NN = [c for c in X_NN.columns if c not in CATS_NN]
NAN_NN = [c for c in NUM_NN if X_NN[c].isna().any() or XT_NN[c].isna().any()]


def target_and_weight(model, rows):
    if model.endswith('_rel'):
        return y[rows] / SCALE[rows], SCALE[rows] ** 2       # 相对力量误差 × scale = kg 误差
    return y[rows], None


def gbdt_fit(model, seed, Xa, ya, wa, Xs=None, ys=None, ws=None, rounds=MAX_ROUNDS):
    """Xs 为 None 时不早停（全量重训）。返回 (模型, 最佳轮数)。"""
    cats = FRAMES[model][2]
    if model in ('lgb', 'lgb_rel'):
        m = lgb.LGBMRegressor(n_estimators=rounds, random_state=seed, n_jobs=-1, **LGB_PARAMS)
        if Xs is None:
            m.fit(Xa, ya, sample_weight=wa)
            return m, rounds
        m.fit(Xa, ya, sample_weight=wa, eval_set=[(Xs, ys)], eval_sample_weight=None if ws is None else [ws],
              callbacks=[lgb.early_stopping(PATIENCE[model], verbose=False)])
        return m, int(m.best_iteration_ or rounds)
    if model == 'xgb':
        if Xs is None:
            m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, **XGB_PARAMS)
            m.fit(Xa, ya, verbose=False)
            return m, rounds
        m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, early_stopping_rounds=PATIENCE['xgb'],
                             **XGB_PARAMS)
        m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
        return m, int(m.best_iteration) + 1
    common = dict(iterations=rounds, loss_function='RMSE', eval_metric='RMSE', random_seed=seed, verbose=0,
                  task_type='GPU', devices='0', allow_writing_files=False, **CB_PARAMS)
    if Xs is None:
        m = catboost.CatBoostRegressor(**common)
        m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa))
        return m, rounds
    m = catboost.CatBoostRegressor(od_type='Iter', od_wait=PATIENCE[model], **common)
    m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa),
          eval_set=catboost.Pool(Xs, ys, cat_features=cats, weight=ws), use_best_model=True)
    return m, int(m.get_best_iteration()) + 1


def gbdt_predict(model, m, best, X):
    if model in ('lgb', 'lgb_rel'):
        return m.predict(X, num_iteration=best)
    if model == 'xgb':
        return m.predict(X, iteration_range=(0, best))
    return m.predict(X)


def nn_frames(*frames):
    """数值缺失用第一个表（外层训练部分）的中位数填补并加缺失标记；类别保持 category。"""
    median = frames[0][NUM_NN].median()
    out = []
    for frame in frames:
        z = frame.copy()
        for c in NAN_NN:
            z[f'{c}_isna'] = z[c].isna().astype('float64')
        z[NUM_NN] = z[NUM_NN].fillna(median).fillna(0.0)
        out.append(z.reset_index(drop=True))
    return out


def tabm_fit_predict(seed, a, s, v, Xt, overrides=None):
    require(TabM_D_Regressor is not None, f'pytabkit 不可用：{PYTABKIT_ERROR}')
    params = dict(TABM_PARAMS, random_state=seed, **(overrides or {}))
    unknown = set(params) - set(inspect.signature(TabM_D_Regressor.__init__).parameters)
    require(not unknown, f'TabM 不支持参数 {sorted(unknown)}')
    Za, Zs, Zv, Zt = nn_frames(X_NN.iloc[a], X_NN.iloc[s], X_NN.iloc[v], Xt)
    mu, sd = y[a].mean(), y[a].std()
    m = TabM_D_Regressor(**params)
    m.fit(Za, (y[a] - mu) / sd, Zs, (y[s] - mu) / sd, cat_col_names=[c for c in Za.columns if c in CATS_NN])
    p_val = np.asarray(m.predict(Zv), dtype='float64').reshape(-1) * sd + mu
    p_test = np.asarray(m.predict(Zt), dtype='float64').reshape(-1) * sd + mu
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return p_val, p_test


def fit_fold(model, seed, a, s, v, test_rows=None, rounds=MAX_ROUNDS, overrides=None):
    """a / s / v：内层拟合 / 内层早停 / 外层验证 的训练行号。返回 (验证预测, 测试预测, 最佳轮数或 None)。"""
    Xtr, Xte, _ = FRAMES[model]
    Xt = Xte if test_rows is None else Xte.iloc[test_rows]
    scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
    if model == 'tabm':
        p_val, p_test = tabm_fit_predict(seed, a, s, v, Xt, overrides)
        return p_val, p_test, None
    ya, wa = target_and_weight(model, a)
    ys, ws = target_and_weight(model, s)
    if wa is not None:
        norm = wa.mean()
        wa, ws = wa / norm, ws / norm
    m, best = gbdt_fit(model, seed, Xtr.iloc[a], ya, wa, Xtr.iloc[s], ys, ws, rounds)
    p_val, p_test = gbdt_predict(model, m, best, Xtr.iloc[v]), gbdt_predict(model, m, best, Xt)
    if model.endswith('_rel'):
        p_val, p_test = np.asarray(p_val) * SCALE[v], np.asarray(p_test) * scale_t
    return np.asarray(p_val, dtype='float64'), np.asarray(p_test, dtype='float64'), best


def fit_full(model, seed, rounds, test_rows=None):
    """全部训练行、固定轮数、不早停。返回测试预测。"""
    Xtr, Xte, _ = FRAMES[model]
    Xt = Xte if test_rows is None else Xte.iloc[test_rows]
    scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
    rows = np.arange(n_train)
    ya, wa = target_and_weight(model, rows)
    if wa is not None:
        wa = wa / wa.mean()
    m, _ = gbdt_fit(model, seed, Xtr, ya, wa, rounds=rounds)
    p_test = gbdt_predict(model, m, rounds, Xt)
    if model.endswith('_rel'):
        p_test = np.asarray(p_test) * scale_t
    return np.asarray(p_test, dtype='float64')



CB_PARAMS_BASE = dict(CB_PARAMS)
CB_PARAMS_LR03 = dict(CB_PARAMS_BASE, learning_rate=0.03)


def with_cb(params, fn, *args, **kwargs):
    """临时替换 CatBoost 参数（gbdt_fit 在调用时读取全局 CB_PARAMS）。"""
    global CB_PARAMS
    saved = CB_PARAMS
    CB_PARAMS = params
    try:
        return fn(*args, **kwargs)
    finally:
        CB_PARAMS = saved


def save_result(stem, arrays, info):
    np.savez(OUTPUT_DIR / f'{stem}.npz', **arrays)
    info = {**info, 'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
    (OUTPUT_DIR / f'{stem}.json').write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')
    return info


def load_cached(stem, spec):
    json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
    if json_path.is_file() and npz_path.is_file():
        saved = json.loads(json_path.read_text(encoding='utf-8'))
        require(saved['spec'] == spec, f'{stem} 的定义已改变；请改 EXPERIMENT_ID')
        return saved
    return None


# 11 的结果必须完整：每个模型 3 个种子 × 5 折，树模型 3 个全量重训
for model in FINAL_MODELS:
    for s in PREV_SEEDS:
        for k in range(1, N_FOLDS + 1):
            require((PREV_DIR / f'{model}__s{s}__fold{k}.npz').is_file(), f'11 缺少 {model} 种子 {s} 第 {k} 折')
        if model != 'tabm':
            require((PREV_DIR / f'{model}__s{s}__full.npz').is_file(), f'11 缺少 {model} 种子 {s} 的全量重训')

# 预检：几秒内确认 cb_rel（学习率 0.03）和 TabM 近全量的代码路径可用
rng = np.random.default_rng(SEED)
probe = rng.choice(n_train, 3000, replace=False)
pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
p_val, p_test, _ = with_cb(CB_PARAMS_LR03, fit_fold, 'cb_rel', SEED, pa, ps, pv, test_rows=np.arange(500), rounds=30)
require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), 'cb_rel 预检失败')
p_full = with_cb(CB_PARAMS_LR03, fit_full, 'cb_rel', SEED, 30, test_rows=np.arange(500))
require(np.isfinite(p_full).all(), 'cb_rel 全量重训预检失败')
TABM_OK = True
try:
    p_val, p_test = tabm_fit_predict(SEED, pa, ps, pv, XT_NN.iloc[:500], overrides={'n_epochs': 3})
    require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), 'TabM 预测无效')
except Exception as exc:
    try:                                               # 与 11 相同：分段线性嵌入不可用时去掉
        TABM_PARAMS = dict(TABM_FALLBACK)
        p_val, p_test = tabm_fit_predict(SEED, pa, ps, pv, XT_NN.iloc[:500], overrides={'n_epochs': 3})
        print('TabM 改用不带嵌入的版本：', repr(exc))
    except Exception as exc2:
        TABM_OK = False
        print('TabM 预检失败，跳过 TabM 近全量：', repr(exc2))
print('预检完成；TabM 近全量：', '可用' if TABM_OK else '跳过')

/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


预检完成；TabM 近全量： 可用


## 7. 学习率检查：cb_rel 学习率 0.03 vs 0.05（种子 42 五折，与 11 同一划分配对）

In [7]:
USE_LR03 = False
if RUN_LR_CHECK:
    lr_rows = []
    for fold in range(1, N_FOLDS + 1):
        stem = f'cb_rel_lr03__s{SEED}__fold{fold}'
        spec = {'model': 'cb_rel', 'params': dict(CB_PARAMS_LR03), 'columns': list(FRAMES['cb_rel'][0].columns)}
        info = load_cached(stem, spec)
        if info is None:
            fit_idx, valid_idx = np.flatnonzero(FOLD_IDS != fold), np.flatnonzero(FOLD_IDS == fold)
            inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)  # = 11
            started = time.perf_counter()
            p_val, _, best = with_cb(CB_PARAMS_LR03, fit_fold, 'cb_rel', SEED, inner_fit, inner_stop, valid_idx,
                                     test_rows=np.arange(10))
            info = save_result(stem, {'valid_idx': valid_idx, 'val': p_val},
                               {'spec': spec, 'fold': fold, 'best_iteration': best,
                                'rmse': rmse(y[valid_idx], p_val), 'seconds': time.perf_counter() - started})
        base = json.loads((PREV_DIR / f'cb_rel__s{SEED}__fold{fold}.json').read_text(encoding='utf-8'))
        lr_rows.append({'fold': fold, 'lr_0.05': base['rmse'], 'lr_0.03': info['rmse'],
                        'delta': info['rmse'] - base['rmse'], 'best_iter_0.05': base['best_iteration'],
                        'best_iter_0.03': info['best_iteration'], 'minutes_0.03': info['seconds'] / 60})
        print(f'fold {fold}: lr 0.05 {base["rmse"]:.3f} → lr 0.03 {info["rmse"]:.3f}')
    lr_table = pd.DataFrame(lr_rows).set_index('fold')
    display(lr_table.round(3))
    lr_table.to_csv(OUTPUT_DIR / 'summary_lr_check.csv')
    USE_LR03 = bool(lr_table['delta'].mean() <= -0.02 and (lr_table['delta'] < 0).sum() >= 4)
print('cb_rel 全量重训使用学习率', 0.03 if USE_LR03 else 0.05)

fold 1: lr 0.05 77.285 → lr 0.03 77.117
fold 2: lr 0.05 77.179 → lr 0.03 77.061
fold 3: lr 0.05 77.550 → lr 0.03 77.493
fold 4: lr 0.05 77.339 → lr 0.03 77.317
fold 5: lr 0.05 76.480 → lr 0.03 76.494


,lr_0.05,lr_0.03,delta,best_iter_0.05,best_iter_0.03,minutes_0.03
fold,,,,,,
1,77.285,77.117,-0.167,3543,3741,2.402
2,77.179,77.061,-0.118,2398,4058,2.613
3,77.550,77.493,-0.057,2263,4949,3.152
4,77.339,77.317,-0.022,2374,2896,1.885
5,76.480,76.494,0.014,2211,2727,1.788


cb_rel 全量重训使用学习率 0.03


## 8. 更多全量重训种子 + TabM 近全量（可续跑）

In [8]:
def prev_best_iterations(model):
    return [json.loads((PREV_DIR / f'{model}__s{s}__fold{k}.json').read_text(encoding='utf-8'))['best_iteration']
            for s in PREV_SEEDS for k in range(1, N_FOLDS + 1)]


def refit_plan(model):
    """(参数, 轮数)：轮数 = CV 平均最佳轮数 × REFIT_FACTOR。"""
    if model == 'cb_rel' and USE_LR03:
        bests = [json.loads((OUTPUT_DIR / f'cb_rel_lr03__s{SEED}__fold{k}.json').read_text(encoding='utf-8'))
                 ['best_iteration'] for k in range(1, N_FOLDS + 1)]
        return CB_PARAMS_LR03, int(round(np.mean(bests) * REFIT_FACTOR))
    return CB_PARAMS_BASE, int(round(np.mean(prev_best_iterations(model)) * REFIT_FACTOR))


for seed in EXTRA_SEEDS:
    for model in EXTRA_MODELS:
        params, rounds = refit_plan(model)
        stem = f'{model}__s{seed}__full'
        spec = {'model': model, 'rounds': rounds, 'columns': list(FRAMES[model][0].columns),
                'cb_params': dict(params) if model in ('catboost', 'cb_rel') else None}
        info = load_cached(stem, spec)
        if info is None:
            started = time.perf_counter()
            p_test = with_cb(params, fit_full, model, seed, rounds)
            require(np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
            info = save_result(stem, {'test': p_test}, {'spec': spec, 'seed': seed,
                                                        'seconds': time.perf_counter() - started})
            how = f'{info["seconds"]:.0f}s'
        else:
            how = 'cache'
        print(f'种子 {seed:3d} 全量重训 {model:8s} 轮数 {rounds}  ({how})')

if TABM_OK:
    for seed in TABM_FULL_SEEDS:
        stem = f'tabm__s{seed}__nearfull'
        spec = {'model': 'tabm', 'params': dict(TABM_PARAMS), 'holdout': TABM_HOLDOUT,
                'columns': list(X_NN.columns)}
        info = load_cached(stem, spec)
        if info is None:
            fit_rows, stop_rows = train_test_split(np.arange(n_train), test_size=TABM_HOLDOUT, random_state=seed)
            started = time.perf_counter()
            p_stop, p_test = tabm_fit_predict(seed, fit_rows, stop_rows, stop_rows, XT_NN)
            require(np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
            info = save_result(stem, {'test': p_test}, {'spec': spec, 'seed': seed,
                                                        'holdout_rmse': rmse(y[stop_rows], p_stop),
                                                        'seconds': time.perf_counter() - started})
        print(f'TabM 近全量 种子 {seed}：早停集 RMSE {info["holdout_rmse"]:.3f}（仅供参考，早停用过它）')

种子  11 全量重训 lgb      轮数 555  (19s)
种子  11 全量重训 xgb      轮数 642  (11s)
种子  11 全量重训 lgb_rel  轮数 515  (18s)
种子  11 全量重训 cb_rel   轮数 4409  (166s)
种子  23 全量重训 lgb      轮数 555  (19s)
种子  23 全量重训 xgb      轮数 642  (10s)
种子  23 全量重训 lgb_rel  轮数 515  (19s)
种子  23 全量重训 cb_rel   轮数 4409  (166s)
种子  57 全量重训 lgb      轮数 555  (19s)
种子  57 全量重训 xgb      轮数 642  (10s)
种子  57 全量重训 lgb_rel  轮数 515  (19s)
种子  57 全量重训 cb_rel   轮数 4409  (166s)
种子  99 全量重训 lgb      轮数 555  (20s)
种子  99 全量重训 xgb      轮数 642  (10s)
种子  99 全量重训 lgb_rel  轮数 515  (19s)
种子  99 全量重训 cb_rel   轮数 4409  (165s)
种子 123 全量重训 lgb      轮数 555  (20s)
种子 123 全量重训 xgb      轮数 642  (10s)
种子 123 全量重训 lgb_rel  轮数 515  (19s)
种子 123 全量重训 cb_rel   轮数 4409  (167s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


TabM 近全量 种子 42：早停集 RMSE 77.684（仅供参考，早停用过它）


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


TabM 近全量 种子 7：早停集 RMSE 75.648（仅供参考，早停用过它）


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


TabM 近全量 种子 2024：早停集 RMSE 77.772（仅供参考，早停用过它）


## 9. 组装测试预测（权重沿用 11）

In [9]:
weights = pd.read_csv(PREV_DIR / 'summary_weights.csv')
NAMES, W = weights['model'].tolist(), weights['weight'].to_numpy(dtype='float64')
NESTED = np.load(PREV_DIR / 'oof_nested_blend.npy')
require(len(NESTED) == n_train and set(NAMES) <= set(FINAL_MODELS), '11 的权重或 OOF 不符')


def fold_average(model):
    return np.mean([np.load(PREV_DIR / f'{model}__s{s}__fold{k}.npz')['test']
                    for s in PREV_SEEDS for k in range(1, N_FOLDS + 1)], axis=0)


rows, PER_MODEL = [], {}
for model in NAMES:
    if model == 'tabm':
        fold_avg = fold_average('tabm')
        near = [np.load(p)['test'] for p in sorted(OUTPUT_DIR.glob('tabm__s*__nearfull.npz'))]
        PER_MODEL[model] = {'seeds': fold_avg, 'seeds_tabmfull': np.mean(near, axis=0) if near else fold_avg}
        rows.append({'model': model, 'test_models': f'{len(near)} 个近全量（另一版用 15 个折模型）'})
    else:
        files = sorted(PREV_DIR.glob(f'{model}__s*__full.npz')) + sorted(OUTPUT_DIR.glob(f'{model}__s*__full.npz'))
        full = np.mean([np.load(p)['test'] for p in files], axis=0)
        PER_MODEL[model] = {'seeds': full, 'seeds_tabmfull': full}
        rows.append({'model': model, 'test_models': f'{len(files)} 个全量重训'})
display(pd.DataFrame(rows).assign(weight=W).set_index('model'))

BLEND_TEST = {f'v2_{kind}': np.column_stack([PER_MODEL[m][kind] for m in NAMES]) @ W
              for kind in ['seeds', 'seeds_tabmfull']}
prev_full = pd.read_csv(PREV_DIR / 'submission_final_full_base.csv')[TARGET].to_numpy()
for kind, pred in BLEND_TEST.items():
    print(f'{kind}：与 11 全量版（未加第二层）的平均绝对差 {np.abs(np.clip(pred, 0, None) - prev_full).mean():.2f} kg')

,test_models,weight
model,,
lgb,8 个全量重训,0.030762
xgb,8 个全量重训,0.200662
lgb_rel,8 个全量重训,0.056515
catboost,3 个全量重训,0.041628
cb_rel,8 个全量重训,0.453233
tabm,3 个近全量（另一版用 15 个折模型）,0.217495


v2_seeds：与 11 全量版（未加第二层）的平均绝对差 1.08 kg
v2_seeds_tabmfull：与 11 全量版（未加第二层）的平均绝对差 1.58 kg


## 10. 第二层：双胞胎 + S3 平滑（与 11 相同，系数由 11 的嵌套 OOF 拟合）→ 导出提交文件

In [10]:
CORE = CORE9


def row_keys(frame, columns):
    return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


def mate_stats(train_keys, query_keys=None):
    table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
    if query_keys is None:                      # 训练行：留一
        keys = pd.Series(train_keys)
        n = keys.map(table['count']).to_numpy() - 1
        s = keys.map(table['sum']).to_numpy() - y
    else:                                       # 测试行：全部训练行
        keys = pd.Series(query_keys)
        n = keys.map(table['count']).fillna(0).to_numpy()
        s = keys.map(table['sum']).fillna(0).to_numpy()
    return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


def twins(query_exact=None, query_core=None):
    n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
    n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
    level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
    return level, np.where(level == 'exact', m_ex, m_co)


def fit_alpha(mask, pred, mate):
    d = mate[mask] - pred[mask]
    denom = float(np.sum(d * d))
    return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


def apply_twins(pred, mate, level, alphas):
    out = pred.copy()
    for lv, a in alphas.items():
        m = level == lv
        out[m] = pred[m] + a * (mate[m] - pred[m])
    return out


LEVEL, MATE = twins()
T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


def text_key(frame, columns):
    out = frame[columns[0]].astype(str)
    for c in columns[1:]:
        out = out + '|' + frame[c].astype(str)
    return out


BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
          'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
          'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


def sums(keys, values):
    table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
    return table['sum'], table['count']


def look(table, keys):
    s, n = table
    return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


def f_exact(k, r):
    ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
    table = sums(ktr, r)
    s, n = look(table, ktr)
    st, nt = look(table, kte)
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


def f_fuzzy(k, r):
    table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
    s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
    for d in (-1, 0, 1):
        a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
        s, n = s + a, n + b
        a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
        st, nt = st + a, nt + b
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


def f_decay(k, r, tau=3.0, max_gap=8):
    base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
    base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
    table = sums(base_tr + YR_TR.astype(str), r)
    s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
    for d in range(-max_gap, max_gap + 1):
        w = float(np.exp(-abs(d) / tau))
        a, b = look(table, base_tr + (YR_TR + d).astype(str))
        s, n = s + w * a, n + w * b
        a, b = look(table, base_te + (YR_TE + d).astype(str))
        st, nt = st + w * a, nt + w * b
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


def f_meet(r, lam=20.0):
    table = sums(MEET_TR, r)
    s, n = look(table, MEET_TR)
    st, nt = look(table, MEET_TE)
    return (s - r) / (n - 1 + lam), st / (nt + lam)


def s3_features(r):
    cols_tr, cols_te = [], []
    for builder in (f_exact, f_fuzzy, f_decay):
        for k in PREFIX:
            a, b = builder(k, r)
            cols_tr.append(a * VALID_TR)
            cols_te.append(b * VALID_TE)
    a, b = f_meet(r)
    cols_tr.append(a)
    cols_te.append(b)
    return np.column_stack(cols_tr), np.column_stack(cols_te)


# 第二层只依赖训练行的 OOF（嵌套融合），对三种测试预测使用同一组 alpha / 系数
alphas = {lv: fit_alpha(LEVEL == lv, NESTED, MATE) for lv in ['exact', 'core']}
p1 = apply_twins(NESTED, MATE, LEVEL, alphas)
p1_cf = NESTED.copy()
for h in (0, 1):
    a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), NESTED, MATE) for lv in ['exact', 'core']}
    m = HALF == h
    p1_cf[m] = apply_twins(NESTED[m], MATE[m], LEVEL[m], a_h)
r1 = y - p1
F, FT = s3_features(r1)
p2_cf = p1_cf.copy()
for h in (0, 1):
    fit, m = HALF != h, HALF == h
    beta_h = np.linalg.lstsq(F[fit], r1[fit], rcond=None)[0]
    p2_cf[m] = p1_cf[m] + F[m] @ beta_h
BETA = np.linalg.lstsq(F, r1, rcond=None)[0]
STAGE2_ADJ = FT @ BETA
stage_table = pd.DataFrame([{'step': '融合（嵌套）', 'oof_rmse': rmse(y, NESTED)},
                            {'step': '+ 双胞胎（交叉拟合）', 'oof_rmse': rmse(y, p1_cf)},
                            {'step': '+ S3 平滑（交叉拟合）', 'oof_rmse': rmse(y, p2_cf)}]).set_index('step')
display(stage_table.round(3))
stage_table.to_csv(OUTPUT_DIR / 'summary_stage2.csv')
print('alpha：', {k: round(v, 3) for k, v in alphas.items()}, '｜测试中有双胞胎的行：', int((T_LEVEL != 'none').sum()),
      '｜S3 平均调整幅度：', round(float(np.abs(STAGE2_ADJ).mean()), 2), 'kg')

for kind, pred in BLEND_TEST.items():
    final = apply_twins(pred, T_MATE, T_LEVEL, alphas) + STAGE2_ADJ
    for stage, values in [('base', pred), ('final', final)]:
        path = OUTPUT_DIR / f'submission_final_{kind}_{stage}.csv'
        pd.DataFrame({'id': test['id'], TARGET: np.clip(values, 0, None)}).to_csv(path, index=False)
        print('写出', path.name)
np.save(OUTPUT_DIR / 'oof_nested_blend.npy', NESTED)
np.save(OUTPUT_DIR / 'oof_final_crossfit.npy', p2_cf)
print('完成。请提交 submission_final_v2_seeds_final.csv 和 submission_final_v2_seeds_tabmfull_final.csv，'
      '与 11 的 76.360 比较；跑完告诉 Claude。')

,oof_rmse
step,
融合（嵌套）,76.395
+ 双胞胎（交叉拟合）,76.172
+ S3 平滑（交叉拟合）,76.044


alpha： {'exact': 0.854, 'core': 0.524} ｜测试中有双胞胎的行： 519 ｜S3 平均调整幅度： 3.71 kg
写出 submission_final_v2_seeds_base.csv
写出 submission_final_v2_seeds_final.csv
写出 submission_final_v2_seeds_tabmfull_base.csv
写出 submission_final_v2_seeds_tabmfull_final.csv
完成。请提交 submission_final_v2_seeds_final.csv 和 submission_final_v2_seeds_tabmfull_final.csv，与 11 的 76.360 比较；跑完告诉 Claude。
